# 05 — Additional data sets
The same benchmark as notebook 04 on two more bivariate-response data sets from the benchmark
suites of Klein et al. (2026, "enb") and Braun et al. (2026, "Bias"):

| Data set | UCI id | n | Response |
|---|---|---|---|
| Energy efficiency (enb) | 242 | 768 | heating load, cooling load |
| Bias correction of temperature forecasts (Bias) | 514 | 7752 (rows with missing values dropped) | next-day max and min temperature |

**Shift design [my choice].** Gan & Liu define a shift only for Parkinson's. Here the same exponential
tilt is applied along the first principal component of the inputs: w(X) = exp[k s(PC1(X))],
k = 0, 2, 4, 6 (k = 0 = no shift).

**Sizes [my choice].** Bias: 1000 training points as for Parkinson's. enb is small, so it uses half of
the data for training and half of the remaining records as test set.

**Weight clipping (`RATIO_CLIP`) and weighted OTCP / GRCP** (added after notebooks 06 and 08):
`RATIO_CLIP = 0.99` caps the ratio at the 99% quantile of the training weights for ALL weighted
methods **[my choice]** → `results/other_datasets_clip99/`; `None` reproduces the original run in
`results/other_datasets/` (plus the two weighted OTCP / GRCP rows).

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath("../src"))
import cpshift as cp
import experiment as ex
import deep

print("device:", deep.DEVICE)

## Configuration

In [ ]:
QUICK = True
RATIO_CLIP = 0.99      # None = no clipping (original run)

BASE = dict(
    SEED=2026, ALPHA=0.10, N_REPS=20, B=1000, SCP_FIT_FRAC=0.5, K_VALUES=[0, 2, 4, 6],
    DEEP_CAL_FRAC=0.5, DEEP_VAL_FRAC=0.2, HIDDEN=64, N_LAYERS=2, LR=1e-3, WEIGHT_DECAY=1e-4,
    BATCH=64, MAX_EPOCHS=500, PATIENCE=30, EPS=1e-3, OT_M=1024, OT_EPS=0.1, GRID_SIDE=150, GRCP_VOL_POINTS=100,
    RATIO_CLIP=RATIO_CLIP,
)
DATASETS = {
    'enb':  dict(uci_id=242, local_file="../data/ENB2012_data.xlsx", N_TRAIN=384, N_TEST=192),
    'bias': dict(uci_id=514, local_file="../data/Bias_correction_ucl.csv", N_TRAIN=1000, N_TEST=1000),
}
CC = dict(WSC_DELTA=0.10, WSC_N_DIR=1000, WSC_GRID=50, CEC_J=10)
if QUICK:
    BASE.update(N_REPS=1, B=200, MAX_EPOCHS=40, K_VALUES=[0, 6], GRCP_VOL_POINTS=20)
    CC.update(WSC_N_DIR=100)

## Data loading
Uses the `ucimlrepo` package (`pip install ucimlrepo`) when available; otherwise reads the file in
`data/` named in `DATASETS`. Non-numeric columns (e.g. the date) and rows with missing values are dropped.

In [ ]:
import io, zipfile, urllib.request

UCI_ZIP = {
    'enb':  "https://archive.ics.uci.edu/static/public/242/energy+efficiency.zip",
    'bias': "https://archive.ics.uci.edu/static/public/514/bias+correction+of+numerical+prediction+model+temperature+forecast.zip",
}

def download_from_uci(name, spec):
    """Download the UCI zip once and save the data file into data/ (skipped if already there)."""
    dest = spec['local_file']
    if os.path.exists(dest):
        return
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    print(f"{name}: downloading {UCI_ZIP[name]}")
    raw = urllib.request.urlopen(UCI_ZIP[name], timeout=120).read()
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        target = os.path.basename(dest)
        members = [m for m in z.namelist() if os.path.basename(m) == target]
        if not members:
            raise FileNotFoundError(f"{target} not found in the zip; it contains {z.namelist()}")
        with open(dest, 'wb') as f:
            f.write(z.read(members[0]))
    print(f"{name}: saved {dest}")

def load(name, spec):
    try:
        from ucimlrepo import fetch_ucirepo
        d = fetch_ucirepo(id=spec['uci_id'])
        Xdf, Ydf = d.data.features, d.data.targets
    except Exception as e:
        print(f"{name}: ucimlrepo failed ({type(e).__name__}); using {spec['local_file']}")
        download_from_uci(name, spec)
        raw = (pd.read_excel(spec['local_file']) if spec['local_file'].endswith('xlsx')
               else pd.read_csv(spec['local_file']))
        tcols = {'enb': ['Y1', 'Y2'], 'bias': ['Next_Tmax', 'Next_Tmin']}[name]
        Ydf, Xdf = raw[tcols], raw.drop(columns=tcols)
    Xdf = Xdf.select_dtypes(include=[np.number])
    both = pd.concat([Xdf, Ydf], axis=1).dropna()
    X, Y = both[Xdf.columns].to_numpy(float), both[Ydf.columns].to_numpy(float)
    assert Y.shape[1] == 2, f"{name}: expected 2 responses, got {list(Ydf.columns)}"
    print(f"{name}: X {X.shape}, Y {Y.shape}, responses {list(Ydf.columns)}")
    return X, Y

DATA = {name: load(name, spec) for name, spec in DATASETS.items()}

## Run

In [ ]:
all_rows = []
for name, (X, Y) in DATA.items():
    cfg = dict(BASE, N_TRAIN=DATASETS[name]['N_TRAIN'], N_TEST=DATASETS[name]['N_TEST'])
    print(f"\n=== {name} ===")
    d = ex.run(X, Y, cp.PCShiftSampler(X), cfg, CC, deep, methods=ex.METHODS_ALL)
    d.insert(0, 'dataset', name)
    all_rows.append(d)
res_df = pd.concat(all_rows, ignore_index=True)
OUT_DIR = "../results/other_datasets" + (f"_clip{round(100 * RATIO_CLIP)}" if RATIO_CLIP else "")
os.makedirs(OUT_DIR, exist_ok=True)
res_df.to_csv(f"{OUT_DIR}/results_all_reps.csv", index=False)

## Tables (per data set)

In [ ]:
summaries = []
for name in DATA:
    by_k, tables = ex.summarize(res_df[res_df.dataset == name], order=ex.METHODS_ALL)
    by_k.insert(0, 'dataset', name)
    summaries.append(by_k)
    for col in ['coverage', 'volume', 'WSC']:
        print(f"\n{name} — {col.upper()} by k\n", tables[col].to_string())
summ = pd.concat(summaries, ignore_index=True)
summ.to_csv(f"{OUT_DIR}/summary_by_k.csv", index=False)

## Figure

In [ ]:
fig, ax = plt.subplots(len(DATA), 3, figsize=(13, 3.6 * len(DATA)), squeeze=False)
for r, name in enumerate(DATA):
    s0 = summ[summ.dataset == name]
    for m in ex.METHODS:
        s = s0[s0.method == m]
        ls = '-' if 'ours' in m or m in ('EUc-LR', 'WCP-L2-LR') else '--'
        ax[r, 0].plot(s.k, s.coverage, marker='o', ls=ls, label=m if r == 0 else None)
        ax[r, 1].plot(s.k, s.volume, marker='o', ls=ls)
        ax[r, 2].plot(s.k, s.WSC, marker='o', ls=ls)
    ax[r, 0].axhline(90, color='k', lw=1, ls=':')
    ax[r, 0].set_ylabel(f'{name}: coverage (%)'); ax[r, 1].set_ylabel('region area'); ax[r, 2].set_ylabel('WSC (%)')
    ax[r, 1].set_yscale('log')
for x in ax[-1]:
    x.set_xlabel('shift intensity $k$')
fig.legend(loc='lower center', ncol=5, fontsize=8, frameon=False, bbox_to_anchor=(0.5, -0.05))
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig_other_datasets.png", dpi=200, bbox_inches='tight'); plt.show()

In [ ]:
print("Files saved in", os.path.abspath(OUT_DIR), ":", sorted(os.listdir(OUT_DIR)))